## Preprocessing and Pipelines


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


Create mixed data with missing values


In [2]:
data = pd.DataFrame(
    {
        'age': [21, 34, np.nan, 45, 29, 38, np.nan, 41],
        'income': [30000, 50000, 42000, np.nan, 39000, 61000, 52000, 58000],
        'city': ['A', 'B', 'A', 'C', 'B', 'C', 'A', np.nan],
        'target': [0, 1, 0, 1, 0, 1, 0, 1],
    }
)

X = data.drop('target', axis=1)
y = data['target']
print(X)
print(y.values)


    age   income city
0  21.0  30000.0    A
1  34.0  50000.0    B
2   NaN  42000.0    A
3  45.0      NaN    C
4  29.0  39000.0    B
5  38.0  61000.0    C
6   NaN  52000.0    A
7  41.0  58000.0  NaN
[0 1 0 1 0 1 0 1]


Build preprocessing pipeline


In [3]:
numeric_features = ['age', 'income']
categorical_features = ['city']

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)


Train model using full pipeline


In [4]:
clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_estimators=100)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

clf.fit(X_train, y_train)
print(f"Pipeline accuracy: {clf.score(X_test, y_test):.4f}")


Pipeline accuracy: 0.0000


In [5]:
scores = cross_val_score(clf, X, y, cv=4)
print(f"Cross-validation scores: {scores}")
print(f"Mean CV score: {scores.mean():.4f}")


Cross-validation scores: [0.5 1.  0.5 1. ]
Mean CV score: 0.7500
